**Contributed workflow — live GeoLab validation is still required.**

Read [README.md](README.md) first. Use a dedicated database and a small date subset. These notebooks can modify database collections and write S3 objects. No CSS data or credentials are distributed here.

# Read Waveform Data
As the name suggests this notebook constructs a working data set for a year set in box 2.   This notebook needs to be run after running load_anfdata and s3indexing.    The output is two things:   (1) a set of day pickle files written to Earthscope's scratch bucket, (2) a new collection called "s3_read_failures".   The pickle files can be transferred to local store and loaded easily into mongoDB with a read - save_data loop over all files (or map driven by the list of pickle files).   Warning:  those pickle files are day files, not event files.   To sort the data into common source ensembles is best done after loading with appropriate queries.   (2) is designed to make it possible to retry failures.   That is, each document in that collection has the original document sent to the s3 reader to pull data that failed.   

First, the stock starting incantation.

In [ ]:
from mspasspy.client import Client
mspass_client=Client()
db=mspass_client.get_database("ANF48")
dask_client = mspass_client.get_scheduler()

In [ ]:
# this workflow is run in yearly blocks.   Change this to change year being handled this run
year = 2014
first_julday = 1
days_to_process = 3  # Set to None only after validating a representative subset.

In [ ]:
# this isn't essential but convenient for monitoring a long run with dask diagnostics
dask_client

Fetch only data suitable for teleseismic P wave processing for receiver functions.  Use the "delta" attribute from assoc to limit data to epicentral distances between 30 and 95.   Note in this notebook that does nothing as we already filtered input that way, but shows an alternative way to do tha with MongoDB.

In [ ]:
from obspy import UTCDateTime
starttime = UTCDateTime(year=year,julday=first_julday)
endtime = UTCDateTime(year=year+1,month=1,day=1)
if days_to_process is not None:
    endtime = min(endtime, starttime + 86400 * days_to_process)
query = {"$and" : [
                    {"delta" : {"$gte" : 30.0, "$lte" : 95.0}},
                    {"Ptime" : {"$gte" : starttime.timestamp, "$lt" : endtime.timestamp}},
                  ] }

cursor=db.arrival_css30.find(query)
doclist = list(cursor)
if not doclist:
    raise ValueError("No arrivals in this date/distance subset; choose a populated interval.")
print("Size of arrival table loaded = ",len(doclist))  # validate this is reasonable

In [ ]:
# reader we use requires a DataFrame as input 
import pandas as pd
df = pd.DataFrame(doclist)
print(df)

The reader function we will use below requires the DataFrame key "time" to define the reference time it is to use to define waveform segments.   For clarity I chose to set the arrival time from the ANF pick table to "Ptime".   Hence, we need this trivial patch from the pandas API.

In [ ]:
df = df.rename(columns={"Ptime" : "time"})

In [ ]:
del doclist

## Build Parallel Input
We need to define a series of functions needed to drive this workflow.  These may be moved to a module but for now the notebook is preferable as there are likely residual bugs that are way easier to fix if they are inside this notebook.


In [ ]:
import s3segment_reader as geos3

With those functions we can now create a (perhaps overcomplicated) data structure to drive parallel processing.   This function is run serial because it runs fast enough that a parallel construct is unnecessary.

In [ ]:
data_list,error_list = geos3.build_day_processing_blocks(df,
                               -240,
                               300,
                               db,
                               auxkeys=["evid","orid","arid","iphase","delta","pick_channel"],
                               )

The outputs (data_list and error_list) are rather complicated.  Be aware the v1 copies of this notebook have a debug section that clarifies that content.   For this production version that has been removed.

We are done with the DataFrame so release it to reduce memory use.

In [ ]:
del df

This plugin is required for the parallel job below to work.   It instantiates a memory resident client on each worker.

In [ ]:
dask_client.upload_file("s3_worker_plugin.py")

from s3_worker_plugin import S3Worker
s3worker = S3Worker()
dask_client.register_plugin(s3worker)
# This registers the named "s3client" plugin. Restart workers when updating modules.

This box is needed for both serial and parallel processing. The parallel version needs the session object for the completion function that writes pickle files.   Kind of confusing but we need this client for the completion function because it runs under the control of the python process connected to this notebook.

In [ ]:
BUCKET = "earthscope-mseed-res-na3mtd4fq5kz7pntcyr1uh46use2a--ol-s3"


The parallel job below had a serial test found in the version 1 notebooks.   I deleted it here as it was baggage for production.

This notebook does the writer as a completion function with sliding_window_pipeline.  We give it a dbname and have it fetch db with fetch_db_handle for more flexibility but the expectation is database activity will be limited to the process running the master script - norm for a completion function.   For GeoLab this doesn't seem to be a bottleneck because only there is currently a limit of 4 cpus and the work done to crack the miniseed files and extract segments seems to keep that many cpus busy.


In [ ]:
from s3_day_workflow import save_jday_outputs, count_saved


In [ ]:
from mspasspy.workflow import sliding_window_pipeline
import os

# Fetch the scratch bucket path
scratch_bucket = os.environ["SCRATCH_BUCKET"]


In [ ]:
# The way I did the above requires I strip component 1 from the day list to drive the processing.   
process_docs=list()
for i in range(len(data_list)):
    process_docs.append(data_list[i][1])

In [ ]:
# worth recording this before starting
len(process_docs)

In [ ]:
# we need to merge these to run sliding_window_pipeline
def read_and_merge(doclist,bucket=BUCKET):
    enslist=list()
    for doc in doclist:
        ens = geos3.get_s3_segments(doc,bucket=bucket,detrend_type="simple")
        enslist.append(ens)
    return enslist
        

In [ ]:
# this is needed to give workers access to functions in the local s3segement_reader.py file
dask_client.upload_file("s3segment_reader.py")

In [ ]:
import time
import s3fs
s3fshandle = s3fs.S3FileSystem()
t0=time.time()
print("Submitting retrieve to dask cluster")
out = sliding_window_pipeline(process_docs,
                              read_and_merge,
                              dask_client,
                              pfunc_kwargs={"bucket" : BUCKET},
                              completion_function=save_jday_outputs,
                              cfunc_args=[db,s3fshandle,scratch_bucket,year],
                              sliding_window_size=1,
                              accumulator=count_saved,
                              retain_results=False,
                              verbose=True,
                              progress_report_interval=10,
                             )
t=time.time()
print("Elapsed time=",t-t0)
print("Number of items handled=",len(process_docs))
print("Number of day files successfully saved=", out)


In [ ]:
# important check on data loss
n=db.s3_read_failures.count_documents({})
print("Number of failures=",n)
print("Check contents of s3_read_failures collection if it is not zero")